In [1]:
"""
Week 3 — Data Preprocessing
Produces a cleaned (but NOT yet imputed / encoded / scaled) dataset.

Why no imputation / encoding / scaling here:
Best practice (AVM Data Science Best Practices, Sec. 07 "Leakage Prevention
Must Be Structural") requires every fit-time preprocessing step (imputers,
encoders, scalers, outlier thresholds) to be fit on the training split only,
then applied unchanged to validation/test. Doing that fitting here — before
the chronological split even exists — would leak validation/test
distribution into decisions that are supposed to come from training data
alone. Those steps are wrapped in a scikit-learn Pipeline/ColumnTransformer
in 03_baseline_model.py (week4.py), fit exclusively on each training window.
"""

'\nWeek 3 — Data Preprocessing\nProduces a cleaned (but NOT yet imputed / encoded / scaled) dataset.\n\nWhy no imputation / encoding / scaling here:\nBest practice (AVM Data Science Best Practices, Sec. 07 "Leakage Prevention\nMust Be Structural") requires every fit-time preprocessing step (imputers,\nencoders, scalers, outlier thresholds) to be fit on the training split only,\nthen applied unchanged to validation/test. Doing that fitting here — before\nthe chronological split even exists — would leak validation/test\ndistribution into decisions that are supposed to come from training data\nalone. Those steps are wrapped in a scikit-learn Pipeline/ColumnTransformer\nin 03_baseline_model.py (week4.py), fit exclusively on each training window.\n'

In [2]:
import pandas as pd

In [3]:
# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
df = pd.read_csv("cleaned_single_family_sales.csv", low_memory=False)
print(f"Original shape: {df.shape}")

Original shape: (320506, 83)


In [4]:
# ---------------------------------------------------------------------------
# Select useful columns
# ---------------------------------------------------------------------------
keep_cols = [
    "CloseDate",
    "ClosePrice",             # target

    # numeric
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "GarageSpaces",
    "YearBuilt",
    "ParkingTotal",

    # categorical
    "PropertySubType",
    "City",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN",
]
df = df[keep_cols].copy()

In [5]:
# ---------------------------------------------------------------------------
# Normalize categorical dtypes
# ---------------------------------------------------------------------------
# Source MLS extracts are inconsistent about how they represent the same
# categorical value — e.g. a YN field may come through as real Python
# booleans (True/False) for some rows and as text ("Y"/"N") for others.
# Left alone, that column has genuinely mixed types (bool + str), which
# breaks scikit-learn's OneHotEncoder later (it sorts unique values, and
# Python can't compare bool to str). Casting every non-null categorical
# value to a string now — while leaving actual missing values as NaN, so
# they still get picked up by imputation — makes the column type-consistent
# for the rest of the pipeline.
cat_cols_all = [
    "PropertySubType",
    "City",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN",
]
for col in cat_cols_all:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str))

In [6]:
# Rename columns
df = df.rename(columns={
    "BedroomsTotal": "Bedrooms",
    "BathroomsTotalInteger": "Bathrooms",
    "LotSizeSquareFeet": "LotSize",
})

In [7]:
numeric_cols = [
    "ClosePrice",
    "LivingArea",
    "Bedrooms",
    "Bathrooms",
    "LotSize",
    "GarageSpaces",
    "YearBuilt",
    "ParkingTotal",
]
cat_cols = [
    "PropertySubType",
    "City",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN",
]

In [8]:
# ---------------------------------------------------------------------------
# Convert data types
# ---------------------------------------------------------------------------
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [9]:
# format="mixed" lets pandas parse each value on its own terms instead of
# guessing one global format, which avoids the
# "Could not infer format, so each element will be parsed individually"
# UserWarning that fires when a date column has more than one representation.
df["CloseDate"] = pd.to_datetime(df["CloseDate"], format="mixed", errors="coerce")

In [10]:
def log_drop(df_before, df_after, reason):
    """Print the count and percentage of rows removed by a cleaning step
    (Best Practices Sec. 02: 'Log the count and percentage of records
    removed at each cleaning step — an undocumented drop in rows is a red
    flag a reviewer should be able to catch.')"""
    n_dropped = len(df_before) - len(df_after)
    pct = 100 * n_dropped / len(df_before) if len(df_before) else 0.0
    print(f"  Dropped {n_dropped} rows ({pct:.2f}%) — {reason}")
    return df_after

In [11]:
# ---------------------------------------------------------------------------
# Data quality: remove records that cannot be true (Sec. 02)
# ---------------------------------------------------------------------------
n0 = len(df)
print(f"\nStarting row-level cleaning from {n0} rows")


Starting row-level cleaning from 320506 rows


In [12]:
_before = df
df = df[df["ClosePrice"] > 0]
df = log_drop(_before, df, "ClosePrice <= 0")

  Dropped 3 rows (0.00%) — ClosePrice <= 0


In [13]:
_before = df
df = df.dropna(subset=["ClosePrice", "CloseDate"])
df = log_drop(_before, df, "missing target or CloseDate")

  Dropped 0 rows (0.00%) — missing target or CloseDate


In [14]:
_before = df
df = df.drop_duplicates()
df = log_drop(_before, df, "exact duplicate rows")

  Dropped 86 rows (0.03%) — exact duplicate rows


In [15]:
_before = df
df = df[df["LivingArea"].isna() | (df["LivingArea"] > 0)]
df = log_drop(_before, df, "LivingArea <= 0 (logical impossibility)")

  Dropped 128 rows (0.04%) — LivingArea <= 0 (logical impossibility)


In [16]:
_before = df
df = df[df["Bedrooms"].isna() | (df["Bedrooms"] >= 0)]
df = df[df["Bathrooms"].isna() | (df["Bathrooms"] >= 0)]
df = log_drop(_before, df, "negative Bedrooms/Bathrooms")

  Dropped 0 rows (0.00%) — negative Bedrooms/Bathrooms


In [17]:
print(f"Rows remaining after cleaning: {len(df)} "
      f"({100 * len(df) / n0:.2f}% of original)\n")

Rows remaining after cleaning: 320289 (99.93% of original)



In [18]:
# ASSUMPTION / KNOWN LIMITATION: ListDate is not present in this extract, so
# the "CloseDate earlier than ListDate" check from the best-practices doc
# could not be applied. Likewise, no parcel ID / address field is available
# to catch duplicate listings beyond exact row duplicates. Both are called
# out here rather than left implicit, per Sec. 10 (Documentation).
# ---------------------------------------------------------------------------
# Missingness indicators (Sec. 06)
# Added BEFORE imputation so the flag itself can be predictive
# (e.g., a missing LotSize often means "condo", which is itself informative).
# The target is never flagged or imputed.
# ---------------------------------------------------------------------------
for col in numeric_cols:
    if col == "ClosePrice":
        continue
    if df[col].isna().any():
        df[f"{col}_missing"] = df[col].isna().astype(int)

In [19]:
# ---------------------------------------------------------------------------
# Outlier filtering (Sec. 03) and missing-value imputation / categorical
# encoding / scaling (Sec. 07) are intentionally NOT performed in this
# script. All of them must be *fit* on the training split only, and the
# chronological split (and the tunable training-window length X) doesn't
# exist yet at this point in the pipeline. They are handled in
# 03_baseline_model.py inside a scikit-learn Pipeline/ColumnTransformer
# that is re-fit for each candidate training window.
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Save cleaned output
# ---------------------------------------------------------------------------
df.to_csv("cleaned_single_family_sales_preprocessed.csv", index=False)
print("Saved cleaned_single_family_sales_preprocessed.csv")
print(f"Final cleaned shape: {df.shape}")

Saved cleaned_single_family_sales_preprocessed.csv
Final cleaned shape: (320289, 22)
